In [ ]:

!pip install -q x-transformers==1.16.23 einops==0.6.1 gin-config pytorch-ignite==0.5.0.post2 pytorch-lightning torchmetrics captum
!git clone -q -b development https://github.com/Katja-Jagd/YAIB.git /content/YAIB



In [ ]:

from google.colab import drive
drive.mount("/content/drive")

import sys, os
sys.path.insert(0, "/content/YAIB")
D = "/content/drive/MyDrive/BAT_colab/"
TAG = "_ivEicu"

from importlib.metadata import version
print("x-transformers version:", version("x-transformers"), "(must be 1.16.23)")

needed = [
    "bat_X_scaled.npy", "bat_static_scaled.npy", "bat_mask.npy", "bat_y.npy",
    "bat_test_rows.npy", "healthy_baseline_zscored.npy",
    "bat_pretrained_eicu_mimiciv.ckpt",            # saved there by the training notebook
    f"bat_best_model{TAG}.pt",                     # saved there by the training notebook
    f"bat_test_predictions{TAG}.npy",              # saved there by the training notebook
]
missing = [f for f in needed if not os.path.exists(D + f)]
print("Missing files:", missing if missing else "none")

id_files = [f for f in ["bat_stay_id_order.csv", "tpc_shap_full_testset_stay_ids.csv"] if os.path.exists(D + f)]
print("Stay ID file(s) found:", id_files if id_files else
      "NONE. Upload bat_stay_id_order.csv from your YAIB folder to BAT_colab before going on.")


Mounted at /content/drive
x-transformers version: 1.16.23 (must be 1.16.23)
Missing files: none
Stay ID file(s) found: ['tpc_shap_full_testset_stay_ids.csv']


In [ ]:

import numpy as np
import pandas as pd
import torch
from icu_benchmarks.models.dl_models.bat import BAT, RegressionHead

device = "cuda" if torch.cuda.is_available() else "cpu"
hp = torch.load(D + "bat_pretrained_eicu_mimiciv.ckpt", map_location="cpu", weights_only=False)["hyper_parameters"]

model = BAT(
    input_size=(64, 48, 24),
    value_embed_size=hp["value_embed_size"],   # 64
    layers=hp["layers"],                       # 6
    heads=hp.get("heads", 1),
    dropout=hp["dropout"],
    attn_dropout=hp["attn_dropout"],
    use_mask=False,
    prediction_head=RegressionHead,
    prediction_head_kwargs={"output_dim": 1},
    static_count=4,
)
model.load_state_dict(torch.load(D + f"bat_best_model{TAG}.pt", map_location="cpu", weights_only=False))
model = model.to(device).eval()
print(f"Model loaded on {device}: {hp['layers']} layers, embed size {hp['value_embed_size']}")

use static
Model loaded on cuda: 6 layers, embed size 64


In [ ]:

X = np.load(D + "bat_X_scaled.npy")            # (51835, 24, 48)
static = np.load(D + "bat_static_scaled.npy")
mask = np.load(D + "bat_mask.npy")
y = np.load(D + "bat_y.npy")
test_rows = np.load(D + "bat_test_rows.npy")
healthy_baseline = np.load(D + "healthy_baseline_zscored.npy")   # (48,)
n_patients = len(test_rows)
assert healthy_baseline.shape == (48,), healthy_baseline.shape

# Stay IDs in test-row order
ids = {}
if os.path.exists(D + "bat_stay_id_order.csv"):
    ids["order"] = pd.read_csv(D + "bat_stay_id_order.csv")["stay_id"].values[test_rows]
if os.path.exists(D + "tpc_shap_full_testset_stay_ids.csv"):
    ids["shap"] = pd.read_csv(D + "tpc_shap_full_testset_stay_ids.csv")["stay_id"].values
assert ids, "No stay ID file found, see CELL 2"
if len(ids) == 2:
    assert np.array_equal(ids["order"], ids["shap"]), "STOP: the two stay ID files disagree"
stay_ids_test = next(iter(ids.values()))
assert len(stay_ids_test) == n_patients

def build_tensors(indices, X_src=None):
    X_src = X if X_src is None else X_src
    return (torch.tensor(np.transpose(X_src[indices], (0, 2, 1)), dtype=torch.float32),  # (n, 48, 24)
            torch.tensor(static[indices], dtype=torch.float32),
            torch.tensor(np.tile(np.arange(24, dtype=np.float32), (len(indices), 1))),
            torch.tensor(np.transpose(mask[indices], (0, 2, 1)), dtype=torch.float32))

def predict_rows(rows, X_src=None, batch_size=256):
    out = []
    with torch.no_grad():
        for i in range(0, len(rows), batch_size):
            xb, sb, tb, mb = [t.to(device) for t in build_tensors(rows[i:i + batch_size], X_src)]
            out.append(model(xb, sb, tb, mb).squeeze(-1).cpu().numpy())
    return np.concatenate(out)

check = predict_rows(test_rows)
saved = np.load(D + f"bat_test_predictions{TAG}.npy").ravel()
diff = np.abs(check - saved).max()
print(f"Largest difference from saved test predictions: {diff:.6f} days (must be below 0.001)")
print(f"Test MAE: {np.abs(check - y[test_rows]).mean():.4f} (training notebook said 2.4708)")
assert diff < 1e-3, "STOP: these weights do not reproduce the saved predictions"

Largest difference from saved test predictions: 0.000002 days (must be below 0.001)
Test MAE: 2.4708 (training notebook said 2.4708)


In [ ]:

# Integrated Gradients, healthy baseline
import time
from captum.attr import IntegratedGradients

N_STEPS, BATCH_SIZE, CHECKPOINT_EVERY = 50, 4, 500
ckpt_path = D + f"bat_ig_healthybaseline_progress_checkpoint{TAG}.npz"

all_attributions = np.zeros((n_patients, 48, 24), dtype=np.float32)
all_deltas = np.zeros(n_patients, dtype=np.float32)
done = np.zeros(n_patients, dtype=bool)
if os.path.exists(ckpt_path):
    c = np.load(ckpt_path)
    all_attributions, all_deltas, done = c["attributions"], c["deltas"], c["done"]
    print(f"Resuming: {done.sum()}/{n_patients} already done")

failed = []
start = time.time()
for b0 in range(0, n_patients, BATCH_SIZE):
    b1 = min(b0 + BATCH_SIZE, n_patients)
    if done[b0:b1].all():
        continue
    X_b, s_b, t_b, m_b = [t.to(device) for t in build_tensors(test_rows[b0:b1])]

    def forward_dynamic_only(x_dynamic):
        return model(x_dynamic, s_b, t_b, m_b).squeeze(-1)

    baseline = torch.tensor(np.tile(healthy_baseline.reshape(1, 48, 1), (b1 - b0, 1, 24)),
                            dtype=torch.float32).to(device)
    try:
        attr, delta = IntegratedGradients(forward_dynamic_only).attribute(
            X_b, baselines=baseline, n_steps=N_STEPS,
            return_convergence_delta=True, internal_batch_size=b1 - b0)   # = 4, except the last, smaller batch
        all_attributions[b0:b1] = attr.detach().cpu().numpy()
        all_deltas[b0:b1] = delta.detach().cpu().numpy()
        done[b0:b1] = True
        del attr, delta
    except Exception as e:
        print(f"Batch {b0}-{b1} failed: {e}")
        failed.extend(range(b0, b1))
    del X_b, s_b, t_b, m_b, baseline
    if device == "cuda":
        torch.cuda.empty_cache()

    if b1 % CHECKPOINT_EVERY < BATCH_SIZE or b1 == n_patients:
        np.savez(ckpt_path, attributions=all_attributions, deltas=all_deltas, done=done)
        print(f"  {done.sum()}/{n_patients} done, {(time.time() - start) / 60:.1f} min")

print(f"\nFinished. Failed patients: {len(failed)}   Not done: {(~done).sum()}")
print(f"Mean |convergence delta|: {np.abs(all_deltas[done]).mean():.6f}")
print(f"Max  |convergence delta|: {np.abs(all_deltas[done]).max():.6f}")
print(f"Stays with |delta| > 0.1: {(np.abs(all_deltas[done]) > 0.1).sum()} "
      f"({100 * (np.abs(all_deltas[done]) > 0.1).mean():.2f}%)")

if done.all():
    np.save(D + f"bat_ig_values_full_testset_healthybaseline{TAG}.npy", all_attributions)
    np.save(D + f"bat_ig_convergence_deltas_healthybaseline{TAG}.npy", all_deltas)
    pd.DataFrame({"stay_id": stay_ids_test}).to_csv(
        D + f"bat_ig_full_testset_stay_ids_healthybaseline{TAG}.csv", index=False)
    print(f"Saved: bat_ig_values_full_testset_healthybaseline{TAG}.npy, "
          f"bat_ig_convergence_deltas_healthybaseline{TAG}.npy, "
          f"bat_ig_full_testset_stay_ids_healthybaseline{TAG}.csv")
else:
    print("NOT saved: some patients failed. Re-run this cell to retry only those.")



  500/10367 done, 5.8 min
  1000/10367 done, 11.6 min
  1500/10367 done, 17.5 min
  2000/10367 done, 23.3 min
  2500/10367 done, 29.1 min
  3000/10367 done, 34.9 min
  3500/10367 done, 40.6 min
  4000/10367 done, 46.4 min
  4500/10367 done, 52.1 min
  5000/10367 done, 57.8 min
  5500/10367 done, 63.5 min
  6000/10367 done, 69.3 min
  6500/10367 done, 75.0 min
  7000/10367 done, 80.6 min
  7500/10367 done, 86.3 min
  8000/10367 done, 91.9 min
  8500/10367 done, 97.5 min
  9000/10367 done, 103.1 min
  9500/10367 done, 108.7 min
  10000/10367 done, 114.7 min
  10367/10367 done, 119.1 min

Finished. Failed patients: 0   Not done: 0
Mean |convergence delta|: 0.023992
Max  |convergence delta|: 2.178542
Stays with |delta| > 0.1: 639 (6.16%)
Saved: bat_ig_values_full_testset_healthybaseline_ivEicu.npy, bat_ig_convergence_deltas_healthybaseline_ivEicu.npy, bat_ig_full_testset_stay_ids_healthybaseline_ivEicu.csv


In [ ]:

# Attention entropy
import time
N_T, N_F = 24, 48
MAX_T, MAX_F = np.log(N_T), np.log(N_F)
BATCH_SIZE = 32
encoder = model.model.encoder_class
n_layers = hp["layers"]

def entropy_from_attn(attn):
    return -(attn * torch.log(attn + 1e-12)).sum(dim=-1)

# attn_layers_1 = feature pathway (keys = 48 sensors), attn_layers_2 = temporal
# pathway (keys = 24 hours). Each runs twice per forward pass (first round,
# then the cross round). The cross round is the second call, as in the original.
captured = {"feature": [], "temporal": []}
def make_hook(name):
    def hook(module, inputs, output):
        captured[name].append(output[1].post_softmax_attn.detach())
    return hook
handles = []
for name, stack in [("feature", encoder.attn_layers_1), ("temporal", encoder.attn_layers_2)]:
    attn_blocks = [layer[1] for layer in stack.layers if hasattr(layer[1], "to_q")]
    assert len(attn_blocks) == n_layers, f"{name}: found {len(attn_blocks)} attention layers"
    for blk in attn_blocks:
        handles.append(blk.register_forward_hook(make_hook(name)))

temporal_entropy = np.zeros((n_patients, N_T), dtype=np.float32)   # layer 0, per hour
feature_entropy = np.zeros((n_patients, N_F), dtype=np.float32)    # layer 0, per sensor
t_mean = np.zeros(n_patients, np.float32); f_mean = np.zeros(n_patients, np.float32)
t_var = np.zeros(n_patients, np.float32);  f_var = np.zeros(n_patients, np.float32)
layer_t = np.zeros((n_patients, n_layers), np.float32)   # every layer, cross round
layer_f = np.zeros((n_patients, n_layers), np.float32)

encoder.return_intermediates = True
start = time.time()
try:
    with torch.no_grad():
        for b0 in range(0, n_patients, BATCH_SIZE):
            b1 = min(b0 + BATCH_SIZE, n_patients); n_b = b1 - b0
            X_b, s_b, t_b, m_b = [t.to(device) for t in build_tensors(test_rows[b0:b1])]
            captured["feature"].clear(); captured["temporal"].clear()

            temporal_attn, feature_attn = encoder(X_b, s_b, t_b, m_b)   # original method
            temporal_attn = temporal_attn.squeeze(1).reshape(n_b, N_F, N_T, N_T)
            feature_attn = feature_attn.squeeze(1).reshape(n_b, N_T, N_F, N_F)
            t_ent = entropy_from_attn(temporal_attn).mean(dim=1)   # (n_b, 24)
            f_ent = entropy_from_attn(feature_attn).mean(dim=1)    # (n_b, 48)
            temporal_entropy[b0:b1] = t_ent.cpu().numpy(); feature_entropy[b0:b1] = f_ent.cpu().numpy()
            t_mean[b0:b1] = t_ent.mean(1).cpu().numpy(); f_mean[b0:b1] = f_ent.mean(1).cpu().numpy()
            t_var[b0:b1] = t_ent.var(1).cpu().numpy();   f_var[b0:b1] = f_ent.var(1).cpu().numpy()

            # hooks: 2 rounds x n_layers calls per pathway; the last n_layers are the cross round
            assert len(captured["temporal"]) == len(captured["feature"]) == 2 * n_layers
            cross_t = captured["temporal"][n_layers:]
            cross_f = captured["feature"][n_layers:]
            if b0 == 0:   # prove the hooks match the encoder's own layer 0 output
                assert torch.allclose(cross_t[0].squeeze(1).reshape(n_b, N_F, N_T, N_T), temporal_attn, atol=1e-6)
                assert torch.allclose(cross_f[0].squeeze(1).reshape(n_b, N_T, N_F, N_F), feature_attn, atol=1e-6)
                print("Hook check passed: hook layer 0 matches the encoder's own attention output.")
            for L in range(n_layers):
                layer_t[b0:b1, L] = entropy_from_attn(cross_t[L].squeeze(1).reshape(n_b, N_F, N_T, N_T)).mean(dim=(1, 2)).cpu().numpy()
                layer_f[b0:b1, L] = entropy_from_attn(cross_f[L].squeeze(1).reshape(n_b, N_T, N_F, N_F)).mean(dim=(1, 2)).cpu().numpy()
finally:
    encoder.return_intermediates = False
    for h in handles:
        h.remove()

print(f"\nDone in {(time.time() - start) / 60:.2f} min")
print(f"\n=== TEMPORAL ATTENTION, layer 0 (original method), T={N_T} ===")
print(f"Mean entropy: {t_mean.mean():.4f}   Normalized: {t_mean.mean() / MAX_T:.4f}   "
      f"Mean within-patient variance: {t_var.mean():.6f}")
print(f"=== FEATURE ATTENTION, layer 0 (original method), F={N_F} ===")
print(f"Mean entropy: {f_mean.mean():.4f}   Normalized: {f_mean.mean() / MAX_F:.4f}   "
      f"Mean within-patient variance: {f_var.mean():.6f}")
print("\n=== Normalized entropy by layer (cross round), 1.0 = uniform ===")
for L in range(n_layers):
    print(f"  Layer {L}: temporal {layer_t[:, L].mean() / MAX_T:.4f}   feature {layer_f[:, L].mean() / MAX_F:.4f}")

np.save(D + f"bat_temporal_attn_entropy_per_timestep{TAG}.npy", temporal_entropy)
np.save(D + f"bat_feature_attn_entropy_per_sensor{TAG}.npy", feature_entropy)
pd.DataFrame({
    "stay_id": stay_ids_test,
    "temporal_entropy_mean": t_mean, "temporal_entropy_normalized": t_mean / MAX_T, "temporal_entropy_variance": t_var,
    "feature_entropy_mean": f_mean, "feature_entropy_normalized": f_mean / MAX_F, "feature_entropy_variance": f_var,
}).to_csv(D + f"bat_attention_entropy_summary{TAG}.csv", index=False)
pd.DataFrame({
    "layer": range(n_layers),
    "temporal_entropy_normalized": layer_t.mean(0) / MAX_T,
    "feature_entropy_normalized": layer_f.mean(0) / MAX_F,
}).to_csv(D + f"bat_attention_entropy_by_layer{TAG}.csv", index=False)
print(f"\nSaved: bat_temporal_attn_entropy_per_timestep{TAG}.npy, bat_feature_attn_entropy_per_sensor{TAG}.npy, "
      f"bat_attention_entropy_summary{TAG}.csv, bat_attention_entropy_by_layer{TAG}.csv")



Hook check passed: hook layer 0 matches the encoder's own attention output.

Done in 0.73 min

=== TEMPORAL ATTENTION, layer 0 (original method), T=24 ===
Mean entropy: 2.5721   Normalized: 0.8093   Mean within-patient variance: 0.009453
=== FEATURE ATTENTION, layer 0 (original method), F=48 ===
Mean entropy: 2.1403   Normalized: 0.5529   Mean within-patient variance: 0.562849

=== Normalized entropy by layer (cross round), 1.0 = uniform ===
  Layer 0: temporal 0.8093   feature 0.5529
  Layer 1: temporal 0.9662   feature 0.4179
  Layer 2: temporal 0.9696   feature 0.9155
  Layer 3: temporal 0.9780   feature 0.7650
  Layer 4: temporal 0.9952   feature 0.9473
  Layer 5: temporal 0.9997   feature 0.9830

Saved: bat_temporal_attn_entropy_per_timestep_ivEicu.npy, bat_feature_attn_entropy_per_sensor_ivEicu.npy, bat_attention_entropy_summary_ivEicu.csv, bat_attention_entropy_by_layer_ivEicu.csv


In [ ]:

# Permutation importance
import time
feature_names = [
    "alb", "alp", "alt", "ast", "be", "bicar", "bili", "bili_dir", "bnd", "bun",
    "ca", "cai", "ck", "ckmb", "cl", "crea", "crp", "dbp", "fgn", "fio2",
    "glu", "hgb", "hr", "inr_pt", "k", "lact", "lymph", "map", "mch", "mchc",
    "mcv", "methb", "mg", "na", "neut", "o2sat", "pco2", "ph", "phos", "plt",
    "po2", "ptt", "resp", "sbp", "temp", "tnt", "urine", "wbc"
]
N_REPEATS = 5
partial_path = D + f"permutation_importance_bat_full{TAG}_partial.csv"
final_path = D + f"permutation_importance_bat_full{TAG}.csv"

X_test_raw = X[test_rows]                     # (n, 24, 48), features on axis 2
local_rows = np.arange(n_patients)            # indexes into the test-only arrays below
static_full, mask_full = static, mask
static, mask = static_full[test_rows], mask_full[test_rows]   # build_tensors now indexes test-only arrays
y_test = y[test_rows]

try:
    baseline_preds = predict_rows(local_rows, X_test_raw)
    baseline_mae = np.abs(baseline_preds - y_test).mean()
    print(f"Baseline MAE (unperturbed): {baseline_mae:.4f} days")

    np.random.seed(42)
    shuffles = [[np.random.permutation(n_patients) for _ in range(N_REPEATS)] for _ in feature_names]

    results = pd.read_csv(partial_path).values.tolist() if os.path.exists(partial_path) else []
    done_names = {r[0] for r in results}
    if done_names:
        print(f"Resuming: {len(done_names)}/48 sensors already done")

    start = time.time()
    for f_idx, f_name in enumerate(feature_names):
        if f_name in done_names:
            continue
        incs = []
        for rep in range(N_REPEATS):
            X_perm = X_test_raw.copy()
            X_perm[:, :, f_idx] = X_test_raw[shuffles[f_idx][rep], :, f_idx]
            incs.append(np.abs(predict_rows(local_rows, X_perm) - y_test).mean() - baseline_mae)
        results.append([f_name, float(np.mean(incs)), float(np.std(incs))])
        pd.DataFrame(results, columns=["feature", "mae_increase", "mae_increase_std"]).to_csv(partial_path, index=False)
        print(f"{f_name}: MAE increase = {np.mean(incs):.4f} (+/- {np.std(incs):.4f})   "
              f"[{len(results)}/48, {(time.time() - start) / 60:.1f} min]")
finally:
    static, mask = static_full, mask_full     # restore full arrays for any later cell

res = pd.DataFrame(results, columns=["feature", "mae_increase", "mae_increase_std"])
res = res.sort_values("mae_increase", ascending=False)
res.to_csv(final_path, index=False)
print("\n=== Top 10 (largest MAE increase when scrambled) ===")
print(res.head(10).to_string(index=False))
print(f"\nSaved: permutation_importance_bat_full{TAG}.csv")


Baseline MAE (unperturbed): 2.4708 days
alb: MAE increase = 0.0005 (+/- 0.0002)   [1/48, 1.9 min]
alp: MAE increase = 0.0000 (+/- 0.0001)   [2/48, 3.9 min]
alt: MAE increase = 0.0010 (+/- 0.0002)   [3/48, 5.9 min]
ast: MAE increase = 0.0000 (+/- 0.0001)   [4/48, 7.9 min]
be: MAE increase = 0.0065 (+/- 0.0019)   [5/48, 9.9 min]
bicar: MAE increase = 0.0050 (+/- 0.0009)   [6/48, 11.8 min]
bili: MAE increase = -0.0014 (+/- 0.0002)   [7/48, 13.8 min]
bili_dir: MAE increase = 0.0001 (+/- 0.0001)   [8/48, 15.8 min]
bnd: MAE increase = 0.0000 (+/- 0.0000)   [9/48, 17.8 min]
bun: MAE increase = 0.0226 (+/- 0.0029)   [10/48, 19.7 min]
ca: MAE increase = 0.0012 (+/- 0.0001)   [11/48, 21.7 min]
cai: MAE increase = -0.0001 (+/- 0.0002)   [12/48, 23.7 min]
ck: MAE increase = -0.0011 (+/- 0.0003)   [13/48, 25.7 min]
ckmb: MAE increase = 0.0000 (+/- 0.0000)   [14/48, 27.7 min]
cl: MAE increase = 0.0065 (+/- 0.0007)   [15/48, 29.6 min]
crea: MAE increase = 0.0061 (+/- 0.0008)   [16/48, 31.6 min]
crp: 

In [ ]:

# Stability under small noise
from captum.attr import IntegratedGradients
np.random.seed(42)
sample_rows = np.random.choice(test_rows, size=200, replace=False)
NOISE_STD = 0.03
ig = IntegratedGradients(model)
pred_diffs, attr_shifts = [], []
baseline = torch.tensor(np.tile(healthy_baseline.reshape(1, 48, 1), (1, 1, 24)), dtype=torch.float32).to(device)

for i, row in enumerate(sample_rows):
    X_p = np.transpose(X[row:row + 1], (0, 2, 1))
    m_p = np.transpose(mask[row:row + 1], (0, 2, 1))
    X_t = torch.tensor(X_p, dtype=torch.float32, requires_grad=True).to(device)
    s_t = torch.tensor(static[row:row + 1], dtype=torch.float32).to(device)
    t_t = torch.tensor(np.arange(24, dtype=np.float32)[None, :]).to(device)
    m_t = torch.tensor(m_p, dtype=torch.float32).to(device)

    with torch.no_grad():
        pred_o = model(X_t, s_t, t_t, m_t).item()
    attr_o = ig.attribute(X_t, baselines=baseline, additional_forward_args=(s_t, t_t, m_t),
                          n_steps=50).detach().cpu().numpy()

    noise = np.random.normal(0, NOISE_STD, size=X_p.shape)
    X_n = torch.tensor(X_p + noise * m_p, dtype=torch.float32, requires_grad=True).to(device)
    with torch.no_grad():
        pred_n = model(X_n, s_t, t_t, m_t).item()
    attr_n = ig.attribute(X_n, baselines=baseline, additional_forward_args=(s_t, t_t, m_t),
                          n_steps=50).detach().cpu().numpy()

    pred_diffs.append(abs(pred_n - pred_o))
    attr_shifts.append(np.abs(attr_n - attr_o).sum() / (np.abs(attr_o).sum() + 1e-8))
    if (i + 1) % 50 == 0:
        print(f"  Processed {i + 1}/200 stays...")

pred_diffs, attr_shifts = np.array(pred_diffs), np.array(attr_shifts)
print("\n=== BAT stability check (new checkpoint) ===")
print(f"Mean prediction shift under noise: {pred_diffs.mean():.4f} days (median: {np.median(pred_diffs):.4f})")
print(f"Mean importance shift (fraction of original magnitude): {attr_shifts.mean():.4f} "
      f"(median: {np.median(attr_shifts):.4f})")
np.save(D + f"bat_automation_bias_pred_diffs{TAG}.npy", pred_diffs)
np.save(D + f"bat_automation_bias_attr_shifts{TAG}.npy", attr_shifts)
print(f"Saved: bat_automation_bias_pred_diffs{TAG}.npy, bat_automation_bias_attr_shifts{TAG}.npy")

  Processed 50/200 stays...
  Processed 100/200 stays...
  Processed 150/200 stays...
  Processed 200/200 stays...

=== BAT stability check (new checkpoint) ===
Mean prediction shift under noise: 0.0188 days (median: 0.0085)
Mean importance shift (fraction of original magnitude): 0.0390 (median: 0.0380)
Saved: bat_automation_bias_pred_diffs_ivEicu.npy, bat_automation_bias_attr_shifts_ivEicu.npy
